# AutoGluon — Landscape of Capabilities
### Part 2

AutoGluon is an open-source AutoML toolkit from AWS. With a few lines you get strong models
through automatic preprocessing, model training, hyperparameter tuning, and **multi-layer stack
ensembling**. This notebook tours the breadth of what AutoGluon can do.

**Modules covered:**
1. `TabularPredictor` — classification & regression on tabular data
2. The model zoo + leaderboard
3. Feature importance
4. `MultiModalPredictor` — text / image (overview, optional run)
5. `TimeSeriesPredictor` — forecasting (overview)
6. Presets, time limits, and quality trade-offs

> Runtime: CPU works; GPU helps multimodal. `Runtime → Run all`.

In [ ]:
# AutoGluon install (Colab). Takes a few minutes.
!pip install -q autogluon.tabular[all]
print("Installed. If Colab asks to restart runtime, do so then Run all again.")

In [ ]:
import pandas as pd
from autogluon.tabular import TabularDataset, TabularPredictor
import autogluon
print("AutoGluon version:", autogluon.__version__)

## 1. Tabular classification — the famous Adult Income dataset

Predict whether income exceeds \$50K/yr. AutoGluon reads a dataframe and a label column — that's it.

In [ ]:
train_url = "https://autogluon.s3.amazonaws.com/datasets/Inc/train.csv"
test_url  = "https://autogluon.s3.amazonaws.com/datasets/Inc/test.csv"
train_data = TabularDataset(train_url).sample(n=2000, random_state=0)
test_data  = TabularDataset(test_url)
label = "class"
print("Shape:", train_data.shape)
train_data.head()

In [ ]:
predictor = TabularPredictor(label=label, eval_metric="accuracy").fit(
    train_data,
    time_limit=120,            # seconds; keep it short for the demo
    presets="medium_quality",  # 'best_quality' trains longer, stronger ensembles
)

## 2. Leaderboard — the whole model zoo at a glance

AutoGluon trained many models (trees, GBMs, neural nets) plus a stacked ensemble. The leaderboard
ranks them on held-out validation data.

In [ ]:
lb = predictor.leaderboard(test_data, silent=True)
lb

In [ ]:
# Evaluate the best model on the test set
perf = predictor.evaluate(test_data)
print(perf)

## 3. Predictions & probabilities

In [ ]:
preds = predictor.predict(test_data.drop(columns=[label]))
proba = predictor.predict_proba(test_data.drop(columns=[label]))
print(preds.head())
proba.head()

## 4. Feature importance

Permutation feature importance tells you which columns drive predictions.

In [ ]:
fi = predictor.feature_importance(test_data)
fi

## 5. Regression — same API, numeric target

AutoGluon auto-detects the problem type from the label. Here we predict a numeric column.

In [ ]:
from sklearn.datasets import fetch_california_housing
cal = fetch_california_housing(as_frame=True).frame
cal_train = cal.sample(frac=0.8, random_state=0)
cal_test  = cal.drop(cal_train.index)

reg = TabularPredictor(label="MedHouseVal", eval_metric="r2").fit(
    cal_train, time_limit=90, presets="medium_quality")
reg.leaderboard(cal_test, silent=True)

In [ ]:
print("Regression test performance:", reg.evaluate(cal_test))

## 6. Other AutoGluon predictors (overview)

AutoGluon spans more than tabular data:

- **`MultiModalPredictor`** — text classification, image classification, and mixed
  text+image+tabular. One `fit()` handles tokenization, backbones, and fine-tuning.
  ```python
  from autogluon.multimodal import MultiModalPredictor
  predictor = MultiModalPredictor(label="label").fit(train_df)
  ```
- **`TimeSeriesPredictor`** — probabilistic forecasting with an ensemble of statistical and
  deep models (DeepAR, PatchTST, etc.).
  ```python
  from autogluon.timeseries import TimeSeriesPredictor
  predictor = TimeSeriesPredictor(prediction_length=48).fit(ts_data)
  ```

These require larger downloads/GPU; the tabular workflow above is the core of AutoGluon's AutoML.

## 7. Presets & time budget

| Preset | Behavior |
|--------|----------|
| `best_quality` | Longest training, strongest stacked ensemble, best accuracy |
| `high_quality` | Strong, deployable, good accuracy/latency balance |
| `good_quality` | Faster, smaller models |
| `medium_quality` | Fast demos / baselines (used here) |

`time_limit` caps wall-clock training. Increase both for real projects.

## Summary
AutoGluon delivers competitive models with almost no manual tuning: hand it a dataframe + a label,
set a `time_limit`/`preset`, and read the leaderboard. It covers tabular, multimodal, and
time-series problems under one consistent API.